In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shapely

from astropy.table import Table
from pydol.photometry.scripts.cmdtools import gen_CMD, gen_CMD_xcut
from pydol.photometry.scripts.catalog_filter import box, ellipse
from astropy.coordinates import angular_separation
from astropy.modeling import models, fitting
import astropy.units as u
from astropy.io import fits
from astropy.wcs import WCS
from astropy.stats import sigma_clipped_stats
from astropy.stats.biweight import biweight_location, biweight_midvariance,biweight_scale
from astropy.visualization import simple_norm

import matplotlib as mpl
from matplotlib.ticker import (MultipleLocator, AutoLocator, AutoMinorLocator)
import seaborn as sb
import json

from glob import glob

In [ ]:
# Minimalistic seaborn style
sb.set_theme(style="white")

mpl.rcParams.update({
    "text.usetex": True,                # If using LaTeX for labels
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "axes.labelsize": 25,
    "font.size": 25,
    "legend.fontsize": 10,
    "xtick.labelsize": 30,
    "ytick.labelsize": 30,
    "axes.titlesize": 30,
    "lines.linewidth": 1.0,
    "lines.markersize": 3,
    "figure.dpi": 300,                  # High-quality output
    "savefig.dpi": 300,
    "axes.grid": False,                 # Avoid grids unless needed
    "legend.frameon": False             # No legend frame
})

plt.rcParams['axes.titlesize'] = plt.rcParams['axes.labelsize'] = 30
plt.rcParams['xtick.labelsize'] = plt.rcParams['ytick.labelsize'] = 30


In [ ]:
Av_dict = { 
            'f275w': 2.02499,
            'f336w': 1.67536,
            'f435w': 1.33879,
            'f555w': 1.03065,
            'f814w': 0.59696,
    
            'f090w': 0.583,
            'f115w': 0.419,
            'f150w': 0.287,
            'f200w': 0.195,
    
            'f438w': 1.34148,
            'f606w': 0.90941,
            'f814w': 0.59845
          }

regions_dict = {
            'ngc628' : {'ra'   : 24.1738983, 
                       'dec'  : 15.7836543,
                        'dismod' : 29.81,
                        'Av': 0.19},
    
            'm83' :   {'ra'   : 204.2536827 ,  
                       'dec'  : -29.8655432,                        
                       'dismod' : 28.41,
                        'Av': 0.18166},
    
            'm51' :  {'ra'   : 202.4696435,
                      'dec'  : 47.1952141,
                      'dismod' : 29.67,
                       'Av': 0.09331},
    
            'ngc4449' : {'ra' : 187.046349,
                         'dec' :44.093666,
                         'dismod' : 28.15,
                         'Av': 0.05208}
            }

In [ ]:
def gen_CMD(
    tab,
    df_iso = None,
    filters={'filt1': 'f115w', 'filt2': 'f150w'},
    positions={'ra_col': 'ra', 'dec_col': 'dec', 'ra_cen': 0, 'dec_cen': 0},
    region={'r_in': 0, 'r_out': 24, 'spatial_filter': 'circle','ang': 245.00492},
    extinction={'Av': 0.19, 'Av_': 3, 'Av_x': 2, 'Av_y': 19},
    distance_modulus=29.7415,
    axis_limits={'xlims': [-0.5, 2.5], 'ylims': [18, 28]},
    isochrone_params={'met': 0.02, 'ages': [7., 8., 9.]},
    plot_settings={'alpha': 1, 's': 0.2, 'lw': 3},
    error_settings={ 'mag_err_lim': 0.2, 'show_err_model': False, 'ref_xpos': -0.5},
    kde_contours={'gen_kde': False, 'gen_contours': False},
    other_settings={'ab_dist': True, 'skip_data': False, 'show_err_model':False},
    fig=None,
    ax=None):
    
    """
    Generate a Color-Magnitude Diagram (CMD) with optional KDE or contour overlays.

    Parameters
    ----------
    tab : DataFrame
        Input data table containing magnitudes, positions, and errors for sources.
        
    df_iso : DataFrame, optional
        Isochrone data for overlay.

    filters : dict, optional
        Filters used in CMD. Keys:
        - 'filt1': Primary filter for color calculation.
        - 'filt2': Secondary filter for color calculation.
        - 'filt3': Filter for magnitude axis. Defaults to 'filt2'.

    positions : dict, optional
        Positional parameters. Keys:
        - 'ra_col': RA column name.
        - 'dec_col': DEC column name.
        - 'ra_cen': Central RA (in degrees).
        - 'dec_cen': Central DEC (in degrees).

    region : dict, optional
        Region parameters for filtering sources. Keys:
        - 'r_in': Inner radius for selection (arcseconds) (used for circular filters).
        - 'r_out': Outer radius for selection (arcseconds) (used for circular filters).
        - 'spatial_filter': Type of spatial filtering ('circle', 'box', 'ellipse').
        - 'ang': Orientation angle (used for box or ellipse filters).
        - 'width_in', 'height_in': Inner box dimensions (arcseconds) (used for box filters).
        - 'width_out', 'height_out': Outer box dimensions (arcseconds) (used for box filters).
        - 'a1', 'b1': Inner semi-major and semi-minor axes (arcseconds) (used for ellipse filters).
        - 'a2', 'b2': Outer semi-major and semi-minor axes (arcseconds) (used for ellipse filters).

    extinction : dict, optional
        Extinction parameters. Keys:
        - 'Av': Extinction value.
        - 'Av_': Annotation extinction value.
        - 'Av_x', 'Av_y': Annotation arrow position.

    distance_modulus : float, optional
        Distance modulus for CMD adjustments. Default is 29.7415.

    axis_limits : dict, optional
        Plot axis limits. Keys:
        - 'xlims': Limits for x-axis (color).
        - 'ylims': Limits for y-axis (magnitude).

    isochrone_params : dict, optional
        Isochrone parameters for plotting. Keys:
        - 'met': Metallicity for isochrones.
        - 'label_min': Minimum label value for filtering.
        - 'label_max': Maximum label value for filtering.
        - 'ages': List of log ages to plot.

    plot_settings : dict, optional
        General plot settings. Keys:
        - 'alpha': Transparency for isochrone lines.
        - 's': Marker size for scatter plots.
        - 'lw': Line width for isochrones.

    error_settings : dict, optional
        Settings for error handling and plotting. Keys:
        - 'mag_err_cols': Columns for magnitude errors. Defaults to filter-based columns.
        - 'mag_err_lim': Maximum allowable magnitude error.
        - 'show_err_model': Show error models during plotting.
        - 'ref_xpos': Reference x-position for error bars.

    kde_contours : dict, optional
        Settings for KDE or contour plots. Keys:
        - 'gen_kde': Generate KDE overlay.
        - 'gen_contours': Generate contour overlay.

    other_settings : dict, optional
        Miscellaneous settings. Keys:
        - 'ab_dist': Include absolute distance modulus adjustments.
        - 'skip_data': Skip scatter plot of source data.

    fig : matplotlib.figure.Figure, optional
        Existing figure object. If None, a new figure is created.

    ax : matplotlib.axes.Axes, optional
        Existing axis object. If None, a new axis is created.


    Returns
    -------
    tuple
        (fig, ax, tab) where:
        - fig: The figure object.
        - ax: The axis object.
        - tab: The filtered input data table after spatial and error-based selection.
    """
    
    # Fill in default values for nested dictionaries
    filters.setdefault('filt1','f115w')
    filters.setdefault('filt2','f200w')
    filters.setdefault('filt3', filters['filt2'])
    
    positions.setdefault('ra_col','ra')
    positions.setdefault('dec_col','dec')
    positions.setdefault('ra_cen',0)
    positions.setdefault('dec_cen',0)
    
    region.setdefault('r_in',0)
    region.setdefault('r_out',10)
    region.setdefault('spatial_filter','circle')
    
    extinction.setdefault('Av',0.19)
    extinction.setdefault('Av_',3)
    extinction.setdefault('Av_x',3)
    extinction.setdefault('Av_y',19)
    
    axis_limits.setdefault('xlims', [-0.5, 2.5])
    axis_limits.setdefault('ylims', [18, 28])
    
    isochrone_params.setdefault('label_min', 0)
    isochrone_params.setdefault('label_max', 10)
    isochrone_params.setdefault('met', [0.02])
    isochrone_params.setdefault('age', [7,8,9])
    
    plot_settings.setdefault('Av.fontsize',15)
    plot_settings.setdefault('legend.fontsize',15)
    plot_settings.setdefault('lw',3)
    plot_settings.setdefault('s',0.2)
    plot_settings.setdefault('alpha',1)
    plot_settings.setdefault('print_met',False)
    plot_settings.setdefault('legend.ncols',1)

    plot_settings.setdefault('color','black')
    
    
    error_settings.setdefault('mag_err_cols', [
        f'mag_err_{filters["filt1"].upper()}',
        f'mag_err_{filters["filt2"].upper()}',
        f'mag_err_{filters["filt3"].upper()}',])
    
    error_settings.setdefault('mag_err_lim',0.2)
    error_settings.setdefault('ref_xpos',-0.25)
    
    kde_contours.setdefault('gen_kde',False)
    kde_contours.setdefault('gen_contours',False)
    kde_contours.setdefault('kde_bin',100j)
    kde_contours.setdefault('cmap','jet')
    kde_contours.setdefault('bw', 0.05)
    
    other_settings.setdefault('ab_dist',True)
    other_settings.setdefault('skip_data',False)
    other_settings.setdefault('show_err_model',False)

    # Filter table by magnitude errors
    for col in error_settings['mag_err_cols']:
        tab = tab[tab[col] <= error_settings['mag_err_lim']]

    # Compute angular separation or define square field
    tab['r'] = angular_separation(
        tab[positions['ra_col']] * u.deg,
        tab[positions['dec_col']] * u.deg,
        positions['ra_cen'] * u.deg,
        positions['dec_cen'] * u.deg).to(u.arcsec).value

    if region['spatial_filter']=='circle':
        tab = tab[(tab['r'] >= region['r_in'])
                  & (tab['r'] <= region['r_out'])]
        
    elif region['spatial_filter']=='box':   
        tab = box(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['width_in'] / 3600, region['height_in'] / 3600,
                  region['width_out'] / 3600, region['height_out'] / 3600,
                  region['ang'])

    elif region['spatial_filter']=='ellipse':
        tab = ellipse(tab, positions['ra_col'], positions['dec_col'],
                  positions['ra_cen'], positions['dec_cen'],
                  region['ang'], 
                  region['a1'] / 3600,region['b1'] / 3600,
                  region['a2'] / 3600,region['b2'] / 3600)
    elif region['spatial_filter']=='polygon':
        tab = polygon(tab, positions['ra_col'], positions['dec_col'], region['points'])

    # Compute magnitudes and colors
    x = tab[f'mag_vega_{filters["filt1"].upper()}'] - tab[f'mag_vega_{filters["filt2"].upper()}']
    y = tab[f'mag_vega_{filters["filt3"].upper()}']

    x = x.value.astype(float)
    y = y.value.astype(float)

    # Initialize figure and axis if not provided
    if fig is None or ax is None:
        fig, ax = plt.subplots(figsize=(12, 10))

    # Extinction corrections
    AF1 = Av_dict[filters['filt1']] * extinction['Av']
    AF2 = Av_dict[filters['filt2']] * extinction['Av']
    AF3 = Av_dict[filters['filt3']] * extinction['Av']

    # Kernel density estimation or scatter plot
    tick_color = 'black'
    if kde_contours['gen_kde'] and not kde_contours['gen_contours']:
        xx, yy = np.mgrid[
            axis_limits['xlims'][0]:axis_limits['xlims'][1]:kde_contours['kde_bin'],
            axis_limits['ylims'][0]:axis_limits['ylims'][1]:kde_contours['kde_bin']]
        
        positions = np.vstack([xx.ravel(), yy.ravel()])
        values = np.vstack([x, y])

        kernel = gaussian_kde(values, bw_method=kde_contours['bw'])
        f = np.reshape(kernel(positions), xx.shape)
        tick_color='white'
        norm = simple_norm(f.T, 'log', min_percent=10)
        ax.imshow(f.T, cmap=kde_contours['cmap'], extent=(*axis_limits['xlims'], *axis_limits['ylims']),
                  interpolation='nearest', aspect='auto', norm=norm)

    elif kde_contours['gen_contours']:
        ax.scatter(x, y, s=plot_settings['s'], color='black', label='data')
        cmap_custom = LinearSegmentedColormap.from_list("custom_grey_to_white", ["grey", "white"], N=256)
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, fill=True, cmap=cmap_custom)
        
        sb.kdeplot(x=x, y=y, levels=[0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99],
                   ax=ax, color='black')

    elif not other_settings['skip_data']:
        ax.scatter(x, y, s=plot_settings['s'], color=plot_settings['color'], label='data')
        
    ax.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
    ax.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
    
    ax.tick_params(which='both', length=15,direction="in", 
                   bottom=True, top=True,left=True, width = 3,
                   color=tick_color)
    
    ax.tick_params(which='minor', length=8, width = 3, color=tick_color)
    ax.xaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    ax.yaxis.set_major_locator(AutoLocator())
    ax.yaxis.set_minor_locator(AutoMinorLocator())
    
    # Handle isochrones
        
    age_lin = []
    for i in isochrone_params['ages']:
        if i < 6:
            age_lin.append(f'{np.round(10**i,1)} Myr')
        if i >= 6  and i < 9:
            i -= 6
            age_lin.append(f'{np.round(10**i,1)} Myr')
        elif i >= 9:
            i -= 9
            age_lin.append(f'{np.round(10**i,1)} Gyr')
            
    if df_iso is not None:
        df_iso = df_iso[(df_iso['label']>=isochrone_params['label_min'])
                       & (df_iso['label']<=isochrone_params['label_max'])]
                        
        for i,age in enumerate(isochrone_params['ages']):
            t = df_iso[(np.round(df_iso['logAge'],1) == age)]
            for Z in isochrone_params['met']:
                subset = t[t['Zini'] == Z]
                x_iso = subset[f"{filters['filt1'].upper()}mag"] + AF1 - (
                        subset[f"{filters['filt2'].upper()}mag"] + AF2)
                y_iso = subset[f"{filters['filt3'].upper()}mag"] + AF3 + distance_modulus
                       
                mask = (y_iso.values[1:]- y_iso.values[:-1])<1
                mask = np.array([True] + list(mask))
                mask = np.where(~mask, np.nan, 1)
                
                if len(isochrone_params['met'])>1 or plot_settings['print_met']:
                    label = label=age_lin[i]+ f' {Z}'
                else:
                    label = label=age_lin[i]
                               
                ax.plot(x_iso*mask, y_iso*mask, lw=plot_settings['lw'],
                        label=label,alpha=plot_settings['alpha'])

    # Absolute magnitude
    if other_settings['ab_dist']:
        yticks = ax.get_yticks()
        yticks_n = yticks - distance_modulus - AF3
        
        dy = yticks_n - np.floor(yticks_n)
        ax1 = ax.twinx()  # instantiate a second axes that shares the same x-axis            
        ax1.set_ylabel(r'$M_{' + f"{filters['filt3'].upper()}" + r'}$')  # we already handled the x-label with ax1
        ax1.set_yticks(yticks - dy, np.floor(yticks_n), fontsize=30)
        ax1.yaxis.set_minor_locator(AutoMinorLocator())
        
        ax1.set_xlim(axis_limits['xlims'][0],axis_limits['xlims'][1])
        ax1.set_ylim(axis_limits['ylims'][0],axis_limits['ylims'][1])
        
        ax1.tick_params(which='both', length=15,direction="in",
                        right=True, width = 3, color=tick_color)
        ax1.tick_params(which='minor', length=8, width = 3, color=tick_color)
        
        
        ax1.invert_yaxis()

    # Extinction Vector
    AF1_ = Av_dict[filters['filt1']] * extinction['Av_']
    AF2_ = Av_dict[filters['filt2']] * extinction['Av_']
    AF3_ = Av_dict[filters['filt3']] * extinction['Av_']

    dx = AF1_ - AF2_
    dy = AF3_

    ax.annotate('', xy=(extinction['Av_x'], extinction['Av_y']),
                 xycoords='data',
                 xytext=(extinction['Av_x']+dx, 
                         extinction['Av_y']+dy),
                 textcoords='data',
                 arrowprops=dict(arrowstyle= '<|-',
                                 color=tick_color,
                                 lw=plot_settings['lw'],
                                 ls='-')
               )

    ax.annotate(f"Av = {extinction['Av_']}",
                xy=(extinction['Av_x']-0.1, extinction['Av_y']-0.1)
                ,fontsize=plot_settings['Av.fontsize'],
                color=tick_color)
    
    # Error models
    if not other_settings['skip_data']:
        ref = tab[f"mag_vega_{filters['filt3'].upper()}"]
        ref_new = np.arange(np.ceil(y.min()),np.floor(y.max())+0.5,0.5)

        mag_err1 = tab[error_settings['mag_err_cols'][0]]
        mag_err2 = tab[error_settings['mag_err_cols'][1]]

        if len(error_settings['mag_err_cols'])>2:
            mag_err3 = tab[error_settings['mag_err_cols'][2]]
        else:
            mag_err3 = mag_err2

        col_err = np.sqrt(mag_err1**2 + mag_err2**2)
        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_col = fit(init,ref,col_err)

        init = models.Exponential1D()
        fit = fitting.LevMarLSQFitter()
        model_mag = fit(init,ref,mag_err3)

        x = error_settings['ref_xpos'] + 0*ref_new
        y = ref_new
        yerr = model_mag(ref_new)
        xerr = model_col(ref_new)
        
        if other_settings['show_err_model']:
            plt.show()
            plt.scatter(ref, mag_err3)
            plt.plot(ref_new,yerr,'--r')
            plt.show()
            plt.scatter(ref, col_err)
            plt.plot(ref_new,xerr,'--r')
            plt.show()
        ax.errorbar(x, y, yerr, xerr ,fmt='o', color = 'red', markersize=0.5, capsize=2) 
    
    # Labels, ticks, and legend
    ax.invert_yaxis()
    
    ax.set_xlabel(f"{filters['filt1'].upper()} - {filters['filt2'].upper()}")
    ax.set_ylabel(filters['filt3'].upper())
    ax.legend(fontsize=plot_settings['legend.fontsize'], ncols = plot_settings['legend.ncols'])

    fig.tight_layout()
    
    return fig, ax, tab

# **Evolutionary Tracks**

In [ ]:
fs = glob("../data/evtracks/Z0.008_Y0.263_tracks/*")

dfs = []
for f in fs:
    df = pd.read_fwf(f, sep='  ', skiprows=2)
    df['Mini'] = float(f.split('M')[-1].split('.TAB')[0])
    df['Zini'] = 0.008
    dfs.append(df)

df_ev_input = pd.concat(dfs)

In [ ]:
df_ev = df_ev_input[df_ev_input['Mini']==50]

x = df_ev['LOG_TE']
y = df_ev['LOG_L']

fig, ax = plt.subplots()

ax.plot(x,y)
ax.invert_xaxis()

In [ ]:
df_ev_input['COCOL'] = df_ev_input['XCsup']/df_ev_input['XOsup']

In [ ]:
df_ev_input[['XCEN', 'Zini','AGE', 'Dtime', 'LOG_L', 'LOG_TE', 'Mini','MASS',
       'RATE','YCEN','Xsup',  'Ysup', 'XCsup','XOsup', 'XNsup','XC_CEN', 'XN_CEN', 'XO_CEN','COCOL']].to_csv('../../TRILEGAL/data/Z0.008.dat', sep=' ', index=None)

In [ ]:
df_evs_acs = pd.read_csv('../data/evtracks/ACS_YBC_Z0.008.txt', sep='\s+')
df_evs_jwst = pd.read_csv('../../TRILEGAL/data/Z0.02.dat_jwst', sep='\s+')

In [ ]:
regions_dict['ngc4449']['dismod'], np.round(10*10**(regions_dict['ngc4449']['dismod']/5)/1e6,2), regions_dict['ngc4449']['Av']

In [ ]:
inp = """1.2450331125827816, -10.194860813704498
0.33554083885209707, -9.029978586723768
-0.11479028697571758, -8.173447537473233
0.16777041942604853, -11.325481798715204
1.2273730684326711, -11.873661670235547
1.5717439293598239, -10.417558886509637
1.6070640176600444, -11.942184154175589"""
a = [[float(j) for j in i.split(',')] for i in inp.split("\n")]
col_diff = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['ngc4449']['Av']
corr = np.array([col_diff, 0])
np.round(a-corr,2) 

In [ ]:
trgb_box = np.array([[ 1.22, -5.11],
                   [ 1.14, -4.71],
                   [ 1.34, -4.45],
                   [ 1.44, -4.88]])

rsg_box = np.array([[  1.06,  -7.39],
                   [  1.36,  -7.39],
                   [  1.56,  -8.7 ],
                   [  1.56, -10.43],
                   [  1.23, -10.19]])

rrsg_box = np.array([[  1.56,  -8.7 ],
                   [  1.56, -10.43],
                   [  4.87, -10.61],
                   [  4.82,  -4.9 ]])

ysg_box = np.array([[  1.06,  -7.39],
                   [  1.23, -10.19],
                   [  0.32,  -9.03],
                   [  0.51,  -7.03]])

bsg_box = np.array([[ 0.51, -7.03],
                   [ 0.32, -9.03],
                   [-0.13, -8.17],
                   [-0.21, -6.1 ]])

yms_box = np.array([[  1.23, -10.19],
                   [  0.32,  -9.03],
                   [ -0.13,  -8.17],
                   [  0.16, -11.33],
                   [  1.22, -11.87],
                   [  1.6 , -11.94],
                   [  1.56, -10.42]])
        

col_diff = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['ngc628']['Av']
mag_diff  = regions_dict['ngc628']['dismod'] + (Av_dict['f115w'])*regions_dict['ngc628']['Av']

ngc_628_corr = np.array([col_diff, mag_diff])

trgb_polygon_ngc628 = shapely.Polygon(trgb_box + ngc_628_corr)
rsg_polygon_ngc628  = shapely.Polygon(rsg_box + ngc_628_corr)
rrsg_polygon_ngc628 = shapely.Polygon(rrsg_box + ngc_628_corr)
ysg_polygon_ngc628  = shapely.Polygon(ysg_box + ngc_628_corr)
bsg_polygon_ngc628  = shapely.Polygon(bsg_box + ngc_628_corr)
yms_polygon_ngc628  = shapely.Polygon(yms_box + ngc_628_corr)


In [ ]:
fig, ax = plt.subplots()
ax.plot(*trgb_polygon_ngc628.exterior.xy, color='red',lw=2)
ax.plot(*rrsg_polygon_ngc628.exterior.xy, color='red',lw=2)
ax.plot(*rsg_polygon_ngc628.exterior.xy, color='red',lw=2)
ax.plot(*ysg_polygon_ngc628.exterior.xy, color='red',lw=2)
ax.plot(*bsg_polygon_ngc628.exterior.xy, color='red',lw=2)
ax.plot(*yms_polygon_ngc628.exterior.xy, color='red',lw=2)

ax.invert_yaxis()

In [ ]:
#  0    1    2     3    4   5   6   7
# RSG1 RSG2 RRSG YSG1 YSG2 BL1 BL2 MYSG
Ns = np.array([[1116, 10349, 0, 3146, 7896, 1294, 3262, 79],
               [3458,  6960, 0, 3225, 8495, 1901, 6127, 493],
               [7396,  9627, 0, 474, 1417, 1197, 3422, 649],
               [870,   9922, 0, 496, 1058, 1816, 3838, 379],
               [7944,  9314, 0, 460, 917, 1493, 7346, 274],
               [11056, 10458, 23, 2, 2183, 269, 474, 15]])

zinis = np.array([0.001, 0.004, 0.008, 0.014, 0.02, 0.03])

In [ ]:
df_ev_jwst = pd.read_csv('../data/evtracks/JWST_evtracks_YBC.txt', sep='\s+')

# **NGC 628**

In [ ]:
df_cmd_jwst = pd.read_csv('../data/isochrones_master/cmd_jwst.csv')

In [ ]:
with open('../data/DS9 regions/regions90_ngc628.json') as json_file:
    regions = json.load(json_file)  

regions_dict.update(regions)     

col_diff = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['ngc628']['Av']
mag_diff  = regions_dict['ngc628']['dismod'] + (Av_dict['f115w'])*regions_dict['ngc628']['Av']

ngc_628_corr = np.array([col_diff, mag_diff])

trgb_polygon_ngc628 = shapely.Polygon(trgb_box + ngc_628_corr)
rsg_polygon_ngc628 = shapely.Polygon(rsg_box + ngc_628_corr)

In [ ]:
rs = [20,80,192]

In [ ]:
df_cmd_hst = pd.read_csv('../data/isochrones_master/cmd_hst_acs.csv')

In [ ]:
tab = Table.read('../catalogs/ngc628/RSGs_HST_JWST.fits')

ra_cen = regions_dict['ngc628']['ra']# 
dec_cen = regions_dict['ngc628']['dec']# 

a = 20000
b = 20000
ang = 0

filters = {'filt1':'f435w',
           'filt2':'f814w',
           'filt3':'f814w'}

positions = {'ra_col': 'ra_2',
             'dec_col' : 'dec_2',
             'ra_cen': ra_cen,
             'dec_cen': dec_cen}

region = {'a1':0,
          'b1':0,
          'a2':a,
          'b2':b,
          'ang':ang,
          'spatial_filter': 'ellipse'}

extinction = {'Av'  : 0.19,
              'Av_x': 4,
              'Av_y': 25,
              'Av_' : 1}

axis_limits= {'xlims': [-1, 6], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 7, 7.7, 8,]}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.2}

plot_settings = {'s':3, 'legend.ncols':2, 'alpha':1}

kde_contours = {'gen_kde':False, 'kde_bin' : 300j, 'cmap': 'jet', 'bw': 0.1}

fig, ax = plt.subplots(figsize=(12,10))

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_hst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      kde_contours=kde_contours,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)
#fig.savefig("NGC628_cmd.png", bbox_inches='tight')


In [ ]:
tab = Table.read('../catalogs/ngc628/RSGs_HST_JWST.fits')

ra_cen = regions_dict['ngc628']['ra']# 
dec_cen = regions_dict['ngc628']['dec']# 

a = 20000
b = 20000
ang = 0

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f200w'}

positions = {'ra_col': 'ra_2',
             'dec_col' : 'dec_2',
             'ra_cen': ra_cen,
             'dec_cen': dec_cen}

region = {'a1':0,
          'b1':0,
          'a2':a,
          'b2':b,
          'ang':ang,
          'spatial_filter': 'ellipse'}

extinction = {'Av'  : 0.19,
              'Av_x': 4,
              'Av_y': 25,
              'Av_' : 1}

axis_limits= {'xlims': [-1, 5.2], 
              'ylims': [17, 28]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 7, 7.7, 8,]}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.2}

plot_settings = {'s':3, 'legend.ncols':2, 'alpha':1}

kde_contours = {'gen_kde':False, 'kde_bin' : 300j, 'cmap': 'jet', 'bw': 0.1}

fig, ax = plt.subplots(figsize=(12,10))

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      29.81,
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      kde_contours=kde_contours,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)
#fig.savefig("NGC628_cmd.png", bbox_inches='tight')


In [ ]:
tab = Table.read('../photometry/ngc628/f115w_f150w_f200w_photometry.fits')

dats_r = []
ra_cen  = regions_dict['ngc628']['ra']
dec_cen = regions_dict['ngc628']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

for i in range(len(rs)-1):

    region = {'a1': rs[i],
              'b1': rs[i],
              'a2': rs[i+1],
              'b2': rs[i+1],
              'ang' : 0,
              'spatial_filter': 'ellipse'}
    
    extinction = {'Av'  : regions_dict['ngc628']['Av'],
                  'Av_x': 1.8,
                  'Av_y': 19,
                  'Av_' : 2}
    
    axis_limits= {'xlims': [-0.5, 3], 
                  'ylims': [18, 28]}
    
    isochrone_params={'met': 0.002,
                      'label_min': 0,
                      'label_max': 3,
                      'age': 10.0}
    
    error_settings = {'ref_xpos': -0.4,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}
    
    m = 21.35#f200w[k]
    
    x_cut_settings = {'cmd_xlo' : -1,
                  'cmd_xhi' : 1.6,
                  'cmd_ylo' : 22,
                  'cmd_yhi' : 26,
    
                  'rgb_xlo' : -1,
                  'rgb_xhi' : 1.6,
                  'rgb_ylo' : 21,
                  'rgb_yhi' : 24,
                  'y_lo'    : 20,
                  'y_hi'    : 24.1 ,
                  'x_lo'    : -1,
                  'x_hi'    : 1.6,                  
                  'dy'      : 0.5,
                  'dx'      : 2.,
                  'theta'   : np.radians(-50),
                  'slope'   : 0,
                  'intercept' : 0.6,
                  'fit_rgb' : False,
                  'fit_isochrone' : False,
                  'rgb_yhi' : 25.5,
                  'color'   : 'black'
                  
                  }
        
    fig, ax = plt.subplots(figsize=(12, 12))
    
    _, _, dats, x_val, y_val, y_bins, model_rgb= gen_CMD_xcut(tab=tab,
                                                         df_iso=df_cmd_jwst,
                                                         distance_modulus=regions_dict['ngc628']['dismod'],
                                                         filters=filters,
                                                         positions=positions,
                                                         region=region,
                                                         extinction=extinction,
                                                         axis_limits=axis_limits,
                                                         isochrone_params=isochrone_params,
                                                         error_settings=error_settings,
                                                         plot_settings=plot_settings,
                                                         x_cut_settings=x_cut_settings,
                                                         fig=fig,
                                                         ax=ax)
    
    
    ax.tick_params(which='both', length=20,direction="in", bottom=True, top=True,left=True, right=True)
    ax.tick_params(which='minor', length=15)
    
    dats_r.append(dats)
    fig.savefig(f"massive_stars/NGC628/CMD_r_{rs[i+1]}.png")
    plt.close(fig)

In [ ]:
reds_r = []
blues_r = []
for r, dats in zip(rs[1:], dats_r):
    red = []
    blue = []
    for i, dat in enumerate(dats):
        x = dat[0]
        y = dat[1]
        ind_r = (x>=0.75) &  (x<=1.5)
        ind_b = (x>=0.) &  (x<=.5)
        red.extend(list(y[ind_r]))
        blue.extend(list(y[ind_b]))

    bins = np.arange(20,24,0.5)
    yr, xr = np.histogram(red, bins=bins)
    yb, xb = np.histogram(blue, bins=bins)
    xr = 0.5*(xr[1:] + xr[:-1])
    reds_r.append(yr)
    blues_r.append(yb)

In [ ]:
reds_sim  = []
blues_sim = []
Zs        = [0.02, 0.008]
for z in Zs:
    red = []
    blue = []
    for mag in np.round(np.arange(20,24,0.5),1):
        dat = np.load(f"massive_stars/simulations/col_hist_Z{z}_mag{mag}.npy")
        x = dat[0]
        y = dat[1]
        ind_r = (x>=0.75) &  (x<=1.5)
        ind_b = (x>=0.) &  (x<=.5)
        red.extend(list(y[ind_r]))
        blue.extend(list(y[ind_b]))

    bins = np.arange(20,24,0.5)
    yr, xr = np.histogram(red, bins=bins)
    yb, xb = np.histogram(blue, bins=bins)
    xr = 0.5*(xr[1:] + xr[:-1])
    reds_sim.append(yr)
    blues_sim.append(yb)

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_sim)):
    y = reds_sim[i]
    ax.plot(x,y, lw=3, label=f"""Z = {Zs[i]} """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Red ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/simulations/Red_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(blues_sim)):
    y = blues_sim[i]
    ax.plot(x,y, lw=3, label=f"""Z = {Zs[i]} """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Blue ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/simulations/Blue_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(blues_sim)):
    y = reds_sim[i]/blues_sim[i]
    ax.plot(x,y,lw=3,label=f"""Z = {Zs[i]} """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel('\#Red/\#Blue')
ax.legend()
ax.set_yscale('log')
fig.savefig(f"massive_stars/simulations/Blue_Red.png", bbox_inches='tight');

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]
    ax.plot(x,y, lw=3, label=f"""r = {rs[1:][i]}" """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Red ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/NGC628/Red_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = blues_r[i]
    ax.plot(x,y, lw=3)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Blue ${\rm LF_{F115W}}$')
ax.set_yscale('log');
fig.savefig(f"massive_stars/NGC628/Blue_F115W_LFs.png", bbox_inches='tight');

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]/blues_r[i]
    ax.plot(x,y,lw=1,label=f"""r = {rs[1:][i]}" """)

arr = np.array(reds_sim)/np.array(blues_sim)
for i in range(arr.shape[1]):
    ax.plot([x[i]]*arr.shape[0], arr[:,i],color='black')
    
markers = ['o','D']
for i in range(len(blues_sim)):
    y = reds_sim[i]/blues_sim[i]
    ax.scatter(x,y, s=30, marker=markers[i],lw=3,label=f"""Z = {Zs[i]} """, zorder=300)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel('\#Red/\#Blue')
ax.legend()
ax.set_yscale('log')
fig.savefig(f"massive_stars/NGC628/Blue_Red.png", bbox_inches='tight');

In [ ]:
for r, dats in zip(rs[1:], dats_r):
    fig, ax = plt.subplots(len(dats),1, figsize=(16,30), sharex=True)
    for i, dat in enumerate(dats):
        if len(dat[0])<10:
            bins  = np.arange(x_cut_settings['x_lo'],x_cut_settings['x_hi'], 0.05)
        else:
            bins  = np.arange(dat[0].min(), dat[0].max(), 0.05)
            
        y, x = np.histogram(dat[0], bins=bins, density=True)
        x = 0.5*(x[1:] + x[:-1])
        ax[i].step(x,y, where='mid', lw=3,color='black')

        if r <= 80:
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.02_mag{y_bins[i]}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='red')
        
        else :
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.008_mag{y_bins[i]}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='green')

        ymax = np.nanmax([y.max(), y_.max()])
        if np.isnan(ymax):
            ymax=1/1.5
            
        ax[i].annotate(f'{y_bins[i]}-{y_bins[i+1]}', 
                       (1.3,1.1*ymax), color='red')
    
        ax[i].xaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].yaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
        ax[i].tick_params(which='minor', length=8)
        ax[i].minorticks_on()
        ax[i].set_ylabel('Counts')


        ax[i].set_ylim(0,1.5*ymax)
        ax[i].set_xlim(-0.5,1.7)
        #ax[i].set_yscale('log')

    ax[i].set_xlabel(r'${\rm F115W - F200W}$')
    plt.subplots_adjust(hspace=0)
    fig.savefig(f"massive_stars/NGC628/hist_r_{r}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
hdul = fits.open('../data/NGC628/jw01783-o908_t016_miri_f770w/jw01783-o908_t016_miri_f770w_i2d.fits')
f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

In [ ]:
fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=f770w_wcs)
norm = simple_norm(f770w, 'log', vmin=-0.1, vmax=15, log_a=10)

ax.imshow(f770w, cmap='gray', norm=norm)

x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
img = ax.scatter(x,y,s=50, color='red', transform=ax.get_transform('icrs'), edgecolor='red', linewidth=0.2,
               )
ax.set_xlim(350,3450)
ax.set_ylim(0,1050)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$', fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

fig.savefig("massive_stars/NGC628/RSG_spatial.png", bbox_inches='tight')
plt.close(fig);

# **M51**

In [ ]:
df_cmd_hst = pd.read_csv('../data/isochrones_master/cmd_hst_acs.csv')

In [ ]:
cols = ['ID', 'F115W', 'eF115W', 'F115W-F200W' ,'errJK', 'F555W', 'eF555W', 'F435W-F555W' ,'eBV', 'F555W-F814W' ,'eVI', 'RA' ,'DEC']
df_rsg = pd.read_csv('../photometry/m51/m51_rsg_all_cmd.dat', sep = ' ' , header=None, names=cols)

for col in cols:
    df_rsg[col] = np.float64([np.nan  if i == 'INDEF' else float(i) for i in  df_rsg[col]])

df_rsg['mag_vega_F435W'] = df_rsg['F435W-F555W'] +  df_rsg['F555W'] 
df_rsg['mag_err_F435W']  = np.sqrt(df_rsg['eBV']**2 - df_rsg['eF555W']**2)

df_rsg['mag_vega_F555W'] = df_rsg['F555W'] 
df_rsg['mag_err_F555W']  = df_rsg['eF555W']

df_rsg['mag_vega_F814W'] = df_rsg['F555W'] - df_rsg['F555W-F814W']
df_rsg['mag_err_F814W']  = np.sqrt(df_rsg['eVI']**2 - df_rsg['eF555W']**2)

df_rsg['mag_vega_F115W'] = df_rsg['F115W']
df_rsg['mag_err_F115W']  = df_rsg['eF115W']

df_rsg['mag_vega_F200W'] = df_rsg['F115W'] - df_rsg['F115W-F200W']
df_rsg['mag_err_F200W']  = np.sqrt(df_rsg['errJK']**2 - df_rsg['eF115W']**2)

df_rsg['ra']  = df_rsg['RA']
df_rsg['dec'] = df_rsg['DEC']

In [ ]:
cols = ['ID', 'F115W', 'eF115W', 'F115W-F200W' ,'errJK', 'F555W', 'eF555W', 'F435W-F555W' ,'eBV', 'F555W-F814W' ,'eVI', 'RA' ,'DEC']
df_bsg = pd.read_csv('../photometry/m51/m51_bsg_all_cmd.dat', sep = ' ' , header=None, names=cols)

for col in cols:
    df_bsg[col] = np.float64([np.nan  if i == 'INDEF' else float(i) for i in  df_bsg[col]])

df_bsg['mag_vega_F435W'] = df_bsg['F435W-F555W'] +  df_bsg['F555W'] 
df_bsg['mag_err_F435W']  = np.sqrt(df_bsg['eBV']**2 - df_bsg['eF555W']**2)

df_bsg['mag_vega_F555W'] = df_bsg['F555W'] 
df_bsg['mag_err_F555W']  = df_bsg['eF555W']

df_bsg['mag_vega_F814W'] = df_bsg['F555W'] - df_bsg['F555W-F814W']
df_bsg['mag_err_F814W']  = np.sqrt(df_bsg['eVI']**2 - df_bsg['eF555W']**2)

df_bsg['mag_vega_F115W'] = df_bsg['F115W']
df_bsg['mag_err_F115W']  = df_bsg['eF115W']

df_bsg['mag_vega_F200W'] = df_bsg['F115W'] - df_bsg['F115W-F200W']
df_bsg['mag_err_F200W']  = np.sqrt(df_bsg['errJK']**2 - df_bsg['eF115W']**2)

df_bsg['ra']  = df_bsg['RA']
df_bsg['dec'] = df_bsg['DEC']

In [ ]:
cols = ['ID', 'F115W', 'eF115W', 'F115W-F200W' ,'errJK', 'F555W', 'eF555W', 'F435W-F555W' ,'eBV', 'F555W-F814W' ,'eVI', 'RA' ,'DEC']
df_ysg = pd.read_csv('../photometry/m51/m51_ysg_all_cmd.dat', sep = ' ' , header=None, names=cols)

for col in cols:
    df_ysg[col] = np.float64([np.nan  if i == 'INDEF' else float(i) for i in  df_ysg[col]])

df_ysg['mag_vega_F435W'] = df_ysg['F435W-F555W'] +  df_ysg['F555W'] 
df_ysg['mag_err_F435W']  = np.sqrt(df_ysg['eBV']**2 - df_ysg['eF555W']**2)

df_ysg['mag_vega_F555W'] = df_ysg['F555W'] 
df_ysg['mag_err_F555W']  = df_ysg['eF555W']

df_ysg['mag_vega_F814W'] = df_ysg['F555W'] - df_ysg['F555W-F814W']
df_ysg['mag_err_F814W']  = np.sqrt(df_ysg['eVI']**2 - df_ysg['eF555W']**2)

df_ysg['mag_vega_F115W'] = df_ysg['F115W']
df_ysg['mag_err_F115W']  = df_ysg['eF115W']

df_ysg['mag_vega_F200W'] = df_ysg['F115W'] - df_ysg['F115W-F200W']
df_ysg['mag_err_F200W']  = np.sqrt(df_ysg['errJK']**2 - df_ysg['eF115W']**2)

df_ysg['ra']  = df_ysg['RA']
df_ysg['dec'] = df_ysg['DEC']

In [ ]:
tab = Table.from_pandas(pd.concat([df_rsg, df_ysg, df_bsg] ))

fig, ax = plt.subplots(figsize=(15,12))
ra_cen = regions_dict['m51']['ra']
dec_cen =regions_dict['m51']['dec']

filters = {'filt1':'f555w',
           'filt2':'f814w',
           'filt3':'f555w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : regions_dict['m51']['Av'],
              'Av_x': 1,
              'Av_y': 19,
              'Av_' : 1}

axis_limits= {'xlims': [-1, 2], 
              'ylims': [17, 24.5]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [ 6.7, 6.8, 7,7.4,7.7, 8]}

error_settings = {'ref_xpos': -0.5,
                  'mag_err_lim':0.5}

plot_settings = {'s':5, 'legend.ncols':2, 'alpha':0.7, 'lw':3, 'color' :'black'}


fig,ax, tab1 = gen_CMD(tab, 
                      None,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

for mass in [80, 50,30,20,16]:
    df_ev = df_evs_acs[(df_evs_acs['Mini']==mass)][200:]
    x = df_ev['F555W'] - df_ev['F814W'] +  (Av_dict['f555w']- Av_dict['f814w'])*regions_dict['m51']['Av']
    y = df_ev['F555W'] + regions_dict['m51']['dismod'] + Av_dict['f555w']*regions_dict['m51']['Av'] 
    
    ax.plot(x,y, lw=3, label=f'Mass = {mass}' + r'$M_{\odot}$')

ax.legend(fontsize=20)
#ax.annotate('RSG', (6.8,18.5), weight='bold', fontsize=40, color='red')
#fig.savefig('massive_stars/M51/M51_CMD.png', bbox_inches='tight')

In [ ]:
np.unique(df_evs_acs['Mini'])

In [ ]:
df_ev = df_evs_acs[(df_evs_acs['Mini']==mass) & (df_evs_acs['Label']>1)]
x = df_ev['F435W'] - df_ev['F555W'] +  (Av_dict['f435w']- Av_dict['f555w'])*regions_dict['m51']['Av']
y = df_ev['F555W'] + regions_dict['m51']['dismod'] + Av_dict['f555w']*regions_dict['m51']['Av'] 

In [ ]:
df_ev['Label']

In [ ]:
df_ev = df_evs_acs[(df_evs_acs['Mini']==50) & (df_evs_acs['Label']<6)][200:]
x = df_ev['LOG_TE']
y = df_ev['LOG_L']

x = df_ev['F555W'] - df_ev['F814W'] +  (Av_dict['f555w']- Av_dict['f814w'])*regions_dict['m51']['Av']
y = df_ev['F555W'] + regions_dict['m51']['dismod'] + Av_dict['f555w']*regions_dict['m51']['Av'] 
c = df_ev['Label']
fig, ax = plt.subplots()

ax.plot(x,y)
img = ax.scatter(x,y,c=c, cmap='jet')
plt.colorbar(img, ax=ax)
ax.invert_yaxis()

In [ ]:
tab = Table.from_pandas(df_rsg)

fig, ax = plt.subplots(figsize=(15,12))
ra_cen = regions_dict['m51']['ra']
dec_cen =regions_dict['m51']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : regions_dict['m51']['Av'],
              'Av_x': 3,
              'Av_y': 19,
              'Av_' : 2}

axis_limits= {'xlims': [-3, 4], 
              'ylims': [16, 25]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [ 6.7, 6.8, 7,7.4,7.7, 8]}

error_settings = {'ref_xpos': -2,
                  'mag_err_lim':0.5}

plot_settings = {'s':5, 'legend.ncols':2, 'alpha':0.7, 'lw':3, 'color' :'red'}


fig,ax, tab1 = gen_CMD(tab, 
                      None,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)


tab = Table.from_pandas(df_ysg)

plot_settings = {'s':5, 'legend.ncols':2, 'alpha':0.7, 'lw':3,  'color' :'green'}

fig,ax, tab1 = gen_CMD(tab, 
                      None,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

tab = Table.from_pandas(df_bsg)

plot_settings = {'s':5, 'legend.ncols':2, 'alpha':0.7, 'lw':3,  'color':'blue'}

fig,ax, tab1 = gen_CMD(tab, 
                      None,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

for mass in [200, 150, 100,80,50,20]:
    df_ev = df_evs_jwst[df_evs_acs['Mini']==mass]
    x = df_ev['F115WmagT'] - df_ev['F200WmagT'] +  (Av_dict['f115w']- Av_dict['f200w'])*regions_dict['m51']['Av']
    y = df_ev['F115WmagT'] + regions_dict['m51']['dismod'] + Av_dict['f115w']*regions_dict['m51']['Av'] 
    
    ax.plot(x,y, lw=3, label=f'Mass = {mass}' + r'$M_{\odot}$')

ax.legend(fontsize=20)
#ax.annotate('RSG', (6.8,18.5), weight='bold', fontsize=40, color='red')
#fig.savefig('massive_stars/M51/M51_CMD.png', bbox_inches='tight')

In [ ]:
col_diff = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['m51']['Av']
mag_diff  = regions_dict['m51']['dismod'] + (Av_dict['f115w'])*regions_dict['m51']['Av']

m_51_corr = np.array([col_diff, mag_diff])

trgb_polygon_m51 = shapely.Polygon(trgb_box + m_51_corr)
rsg_polygon_m51  = shapely.Polygon(rsg_box + m_51_corr)
rrsg_polygon_m51 = shapely.Polygon(rrsg_box + m_51_corr)
ysg_polygon_m51  = shapely.Polygon(ysg_box + m_51_corr)
bsg_polygon_m51  = shapely.Polygon(bsg_box + m_51_corr)
yms_polygon_m51  = shapely.Polygon(yms_box + m_51_corr)


In [ ]:
col_diff, mag_diff =  ngc_628_corr- m_51_corr

In [ ]:
mag_diff

In [ ]:
tab = Table.read('../photometry/m51/f115w_f150w_f200w_photometry.fits')

fig, ax = plt.subplots(figsize=(15,12))
ra_cen = regions_dict['m51']['ra']
dec_cen =regions_dict['m51']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : regions_dict['m51']['Av'],
              'Av_x': 1.8,
              'Av_y': 19,
              'Av_' : 2}

axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
              'ylims': [18-mag_diff, 28-mag_diff]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 6.8, 7,7.4,7.7, 8]}

error_settings = {'ref_xpos': -0.4,
                  'mag_err_lim':0.2}

plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}

df_temp = df_cmd_jwst.copy()

df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]

fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 9.4, 10.]
isochrone_params['met'] = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m51']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)


x = tab1['mag_vega_F115W'] - tab1['mag_vega_F200W']
y = tab1['mag_vega_F115W']

points = np.column_stack([x, y])

ind_trgb = shapely.contains_xy(trgb_polygon_m51, points)
ind_rsg = shapely.contains_xy(rsg_polygon_m51, points)
ind_rrsg = shapely.contains_xy(rrsg_polygon_m51, points)
ind_ysg = shapely.contains_xy(ysg_polygon_m51, points)
ind_bsg = shapely.contains_xy(bsg_polygon_m51, points)
ind_yms = shapely.contains_xy(yms_polygon_m51, points)

tab_TRGB = tab1[ind_trgb]
tab_RSG  = tab1[ind_rsg]
tab_RRSG = tab1[ind_rrsg]
tab_YSG  = tab1[ind_ysg]
tab_BSG  = tab1[ind_bsg]
tab_YMS  = tab1[ind_yms]

ax.scatter(x[ind_rsg], y[ind_rsg], color='red', s=3)
ax.scatter(x[ind_rrsg], y[ind_rrsg], color='magenta', s=3)
ax.scatter(x[ind_ysg], y[ind_ysg], color='yellow', s=3)
ax.scatter(x[ind_bsg], y[ind_bsg], color='cyan', s=3)
ax.scatter(x[ind_yms], y[ind_yms], color='purple', s=3)

ax.legend(fontsize=20)
#fig.savefig('massive_stars/M51/M51_CMD.png', bbox_inches='tight')

In [ ]:
tab_RSG.write('m51_tables/RSG.fits', overwrite=True)
tab_RRSG.write('m51_tables/RRSG.fits', overwrite=True)
tab_YSG.write('m51_tables/YSG.fits', overwrite=True)
tab_BSG.write('m51_tables/BSG.fits', overwrite=True)
tab_YMS.write('m51_tables/YMS.fits', overwrite=True)

In [ ]:
text = """# Region file format: DS9 version 4.1
global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1
icrs
"""

# RSG
with open('m51_tables/rsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_RSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=red \n""")

with open('m51_tables/rsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_RSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=red point=circle \n""")

# RRSG
with open('m51_tables/rrsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_RRSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=magenta \n""")

with open('m51_tables/rrsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_RRSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=magenta point=circle \n""")

# YSG
with open('m51_tables/ysg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_YSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=yellow \n""")

with open('m51_tables/ysg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_YSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=yellow point=circle \n""")

# BSG
with open('m51_tables/bsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_BSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=cyan \n""")

with open('m51_tables/bsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_BSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=cyan point=circle \n""")

# YMS
with open('m51_tables/yms_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_YMS:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=purple \n""")

with open('m51_tables/yms_point.reg','w') as f:
    f.writelines(text)
    for row in tab_YMS:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=purple point=circle \n""")

        

In [ ]:
x = tab_YMS['mag_vega_F115W'] - tab_YMS['mag_vega_F200W']
y = tab_YMS['mag_vega_F115W']

fig, ax = plt.subplots()

ax.scatter(x,y)
ax.invert_yaxis()

In [ ]:
tab_RSG = tab1[ind_rsg]
r       = angular_separation(tab_RSG['ra']*u.deg, tab_RSG['dec']*u.deg ,regions_dict['m51']['ra']*u.deg, regions_dict['m51']['dec']*u.deg).to(u.arcsec).value

In [ ]:
r.max()

In [ ]:
x= tab_RSG['ra']
y = tab_RSG['dec']
c= r

img = plt.scatter(x,y,c=c, cmap='tab10',s=1, vmin=0, vmax = 191)
cb = plt.colorbar(img)
cb.set_ticks([0,20,40,50,60,80,100,150,200])

In [ ]:
rs = [30]
ra_cen  = regions_dict['m51']['ra']
dec_cen = regions_dict['m51']['dec']
dr = 1
r=0
while r<200:
    r = rs[-1] + dr
    tab_filt = ellipse(tab_TRGB,'ra','dec', ra_cen,dec_cen,0,rs[-1]/3600,rs[-1]/3600,r/3600/3600)
    while len(tab_filt)<15000 and r<200:
        r+=dr
        tab_filt = ellipse(tab_TRGB,'ra','dec', ra_cen,dec_cen,0,rs[-1]/3600,rs[-1]/3600,r/3600,r/3600)
    if r<200:
        rs.append(r)
    else:
        rs[-1] =200

In [ ]:
rs = [20,80,192]

In [ ]:
tab = Table.read('../photometry/m51/f115w_f150w_f200w_photometry.fits')

dats_r = []
ra_cen  = regions_dict['m51']['ra']
dec_cen = regions_dict['m51']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

for i in range(len(rs)-1):

    region = {'a1': rs[i],
              'b1': rs[i],
              'a2': rs[i+1],
              'b2': rs[i+1],
              'ang' : 0,
              'spatial_filter': 'ellipse'}
    
    extinction = {'Av'  : regions_dict['m51']['Av'],
                  'Av_x': 1.8,
                  'Av_y': 19,
                  'Av_' : 2}
    
    axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
                  'ylims': [18-mag_diff, 28-mag_diff]}
    
    isochrone_params={'met': 0.002,
                      'label_min': 0,
                      'label_max': 3,
                      'age': 10.0}
    
    error_settings = {'ref_xpos': -0.4,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}
    
    m = 21.35#f200w[k]
    
    x_cut_settings = {'cmd_xlo' : -1,
                  'cmd_xhi' : 1.6,
                  'cmd_ylo' : 22,
                  'cmd_yhi' : 26,
    
                  'rgb_xlo' : -1,
                  'rgb_xhi' : 1.6,
                  'rgb_ylo' : 21,
                  'rgb_yhi' : 24,
                  'y_lo'    : 20-mag_diff,
                  'y_hi'    : 24.1-mag_diff ,
                  'x_lo'    : -1,
                  'x_hi'    : 1.6,                  
                  'dy'      : 0.5,
                  'dx'      : 2.,
                  'theta'   : np.radians(-50),
                  'slope'   : 0,
                  'intercept' : 0.6-col_diff,
                  'fit_rgb' : False,
                  'fit_isochrone' : False,
                  'rgb_yhi' : 25.5,
                  'color'   : 'black'
                  
                  }
        
    fig, ax = plt.subplots(figsize=(12, 12))
    
    _, _, dats, x_val, y_val, y_bins, model_rgb= gen_CMD_xcut(tab=tab,
                                                         df_iso=df_cmd_jwst,
                                                         distance_modulus=regions_dict['m51']['dismod'],
                                                         filters=filters,
                                                         positions=positions,
                                                         region=region,
                                                         extinction=extinction,
                                                         axis_limits=axis_limits,
                                                         isochrone_params=isochrone_params,
                                                         error_settings=error_settings,
                                                         plot_settings=plot_settings,
                                                         x_cut_settings=x_cut_settings,
                                                         fig=fig,
                                                         ax=ax)
    
    
    ax.tick_params(which='both', length=20,direction="in", bottom=True, top=True,left=True, right=True)
    ax.tick_params(which='minor', length=15)
    
    dats_r.append(dats)
    fig.savefig(f"massive_stars/M51/CMD_r_{rs[i+1]}.png")
    plt.close(fig)

In [ ]:
reds_r = []
blues_r = []
for r, dats in zip(rs[1:], dats_r):
    red = []
    blue = []
    for i, dat in enumerate(dats):
        x = dat[0]
        y = dat[1]
        ind_r = (x>=0.75-col_diff) &  (x<=1.5-col_diff)
        ind_b = (x>=0.-col_diff) &  (x<=.5-col_diff)
        red.extend(list(y[ind_r]))
        blue.extend(list(y[ind_b]))

    bins = np.arange(20,24,0.5)-mag_diff
    yr, xr = np.histogram(red, bins=bins)
    yb, xb = np.histogram(blue, bins=bins)
    xr = 0.5*(xr[1:] + xr[:-1])
    reds_r.append(yr)
    blues_r.append(yb)

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]
    ax.plot(x,y, lw=3, label=f"""r = {rs[1:][i]}" """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Red ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/M51/Red_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = blues_r[i]
    ax.plot(x,y, lw=3)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Blue ${\rm LF_{F115W}}$')
ax.set_yscale('log');
fig.savefig(f"massive_stars/M51/Blue_F115W_LFs.png", bbox_inches='tight');

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]/blues_r[i]
    ax.plot(x,y,lw=1,label=f"""r = {rs[1:][i]}" """)

arr = np.array(reds_sim)/np.array(blues_sim)
for i in range(arr.shape[1]):
    ax.plot([x[i]]*arr.shape[0], arr[:,i],color='black')
    
markers = ['o','D']
for i in range(len(blues_sim)):
    y = reds_sim[i]/blues_sim[i]
    ax.scatter(x,y, s=30, marker=markers[i],lw=3,label=f"""Z = {Zs[i]} """, zorder=300)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel('\#Red/\#Blue')
ax.legend()
ax.set_yscale('log')
fig.savefig(f"massive_stars/M51/Blue_Red.png", bbox_inches='tight');

In [ ]:
for r, dats in zip(rs[1:], dats_r):
    fig, ax = plt.subplots(len(dats),1, figsize=(16,30), sharex=True)
    for i, dat in enumerate(dats):
        if len(dat[0])<10:
            bins  = np.arange(x_cut_settings['x_lo'],x_cut_settings['x_hi'], 0.05)
        else:
            bins  = np.arange(dat[0].min(), dat[0].max(), 0.05)
            
        y, x = np.histogram(dat[0], bins=bins, density=True)
        x = 0.5*(x[1:] + x[:-1])
        ax[i].step(x,y, where='mid', lw=3,color='black')

        if r <= 80:
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.02_mag{y_bins[i]+mag_diff}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='red')
        
        else :
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.008_mag{y_bins[i]+mag_diff}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='green')

        ymax = np.nanmax([y.max(), y_.max()])
        if np.isnan(ymax):
            ymax=1/1.5
            
        ax[i].annotate(f'{y_bins[i]+mag_diff}-{y_bins[i+1]+mag_diff}', 
                       (1.3,1.1*ymax), color='red')
    
        ax[i].xaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].yaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
        ax[i].tick_params(which='minor', length=8)
        ax[i].minorticks_on()
        ax[i].set_ylabel('Counts')


        ax[i].set_ylim(0,1.5*ymax)
        ax[i].set_xlim(-0.5,1.7)
        #ax[i].set_yscale('log')

    ax[i].set_xlabel(r'${\rm F115W - F200W}$')
    plt.subplots_adjust(hspace=0)
    fig.savefig(f"massive_stars/M51/hist_r_{r}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
hdul = fits.open('../data/M51/jw01783-o007_t018_miri_f770w/jw01783-o007_t018_miri_f770w_i2d.fits')
f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

In [ ]:
fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=f770w_wcs)
norm = simple_norm(f770w, 'log', vmin=-0.1, vmax=15, log_a=10)

ax.imshow(f770w, cmap='gray', norm=norm)

x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
img = ax.scatter(x,y,s=50, color='red', transform=ax.get_transform('icrs'), edgecolor='red', linewidth=0.2)
ax.set_xlim(350,3450)
ax.set_ylim(0,1050)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$', fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

fig.savefig("massive_stars/M51/RSG_spatial.png", bbox_inches='tight')
plt.close(fig);

# **M83**

In [ ]:
col_diff = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['m83']['Av']
mag_diff  = regions_dict['m83']['dismod'] + (Av_dict['f115w'])*regions_dict['m83']['Av']

m_83_corr = np.array([col_diff, mag_diff])

trgb_polygon_m83 = shapely.Polygon(trgb_box + m_83_corr)
rsg_polygon_m83 = shapely.Polygon(rsg_box + m_83_corr)

In [ ]:
col_diff, mag_diff =  ngc_628_corr - m_83_corr

In [ ]:
18-mag_diff

In [ ]:
tab = Table.read('../photometry/m83/f115w_f150w_f200w_photometry.fits')

fig, ax = plt.subplots(figsize=(15,12))
ra_cen = regions_dict['m83']['ra']
dec_cen =regions_dict['m83']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : regions_dict['m83']['Av'],
              'Av_x': 1.8,
              'Av_y': 19,
              'Av_' : 2}

axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
              'ylims': [18-mag_diff, 28-mag_diff]}

isochrone_params={'met': [0.02],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 6.8, 7,7.4,7.7, 8]}

error_settings = {'ref_xpos': -0.4,
                  'mag_err_lim':0.2}

plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}

df_temp = df_cmd_jwst.copy()
df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]

fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m83']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 9.4, 10.]
isochrone_params['met'] = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['m83']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)


x = tab1['mag_vega_F115W'] - tab1['mag_vega_F200W']
y = tab1['mag_vega_F115W']

points = np.column_stack([x, y])

ind_trgb = shapely.contains_xy(trgb_polygon_m83, points)
ind_rsg = shapely.contains_xy(rsg_polygon_m83, points)

tab_TRGB = tab1[ind_trgb]
tab_RSG = tab1[ind_rsg]

#ax.scatter(x[ind_rsg], y[ind_rsg], color='red', s=3)

ax.legend(fontsize=20)
fig.savefig('massive_stars/M83/M83_CMD.png', bbox_inches='tight')

In [ ]:
tab_RSG = tab1[ind_rsg]
r       = angular_separation(tab_RSG['ra']*u.deg, tab_RSG['dec']*u.deg ,regions_dict['m83']['ra']*u.deg, regions_dict['m83']['dec']*u.deg).to(u.arcsec).value

In [ ]:
r.max()

In [ ]:
x = tab_RSG['ra']
y = tab_RSG['dec']
c = r

img = plt.scatter(x,y,c=c, cmap='tab10',s=1, vmin=0, vmax = 191)
cb = plt.colorbar(img)
cb.set_ticks([0,20,40,50,60,80,100,150,200])

In [ ]:
rs = [20,80,198]

In [ ]:
tab     = Table.read('../photometry/m83/f115w_f150w_f200w_photometry.fits')

dats_r  = []
ra_cen  = regions_dict['m83']['ra']
dec_cen = regions_dict['m83']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

for i in range(len(rs)-1):

    region = {'a1': rs[i],
              'b1': rs[i],
              'a2': rs[i+1],
              'b2': rs[i+1],
              'ang' : 0,
              'spatial_filter': 'ellipse'}
    
    extinction = {'Av'  : regions_dict['m51']['Av'],
                  'Av_x': 1.8,
                  'Av_y': 19,
                  'Av_' : 2}
    
    axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
                  'ylims': [18-mag_diff, 28-mag_diff]}
    
    isochrone_params={'met': 0.002,
                      'label_min': 0,
                      'label_max': 3,
                      'age': 10.0}
    
    error_settings = {'ref_xpos': -0.4,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}
    
    m = 21.35#f200w[k]
    
    x_cut_settings = {'cmd_xlo' : -1,
                  'cmd_xhi' : 1.6,
                  'cmd_ylo' : 22,
                  'cmd_yhi' : 26,
    
                  'rgb_xlo' : -1,
                  'rgb_xhi' : 1.6,
                  'rgb_ylo' : 21,
                  'rgb_yhi' : 24,
                  'y_lo'    : 20  - mag_diff,
                  'y_hi'    : 24.1- mag_diff ,
                  'x_lo'    : -1,
                  'x_hi'    : 1.6,                  
                  'dy'      : 0.5,
                  'dx'      : 2.,
                  'theta'   : np.radians(-50),
                  'slope'   : 0,
                  'intercept' : 0.6-col_diff,
                  'fit_rgb' : False,
                  'fit_isochrone' : False,
                  'rgb_yhi' : 25.5,
                  'color'   : 'black'
                  
                  }
        
    fig, ax = plt.subplots(figsize=(12, 12))
    
    _, _, dats, x_val, y_val, y_bins, model_rgb= gen_CMD_xcut(tab=tab,
                                                         df_iso=df_cmd_jwst,
                                                         distance_modulus=regions_dict['m83']['dismod'],
                                                         filters=filters,
                                                         positions=positions,
                                                         region=region,
                                                         extinction=extinction,
                                                         axis_limits=axis_limits,
                                                         isochrone_params=isochrone_params,
                                                         error_settings=error_settings,
                                                         plot_settings=plot_settings,
                                                         x_cut_settings=x_cut_settings,
                                                         fig=fig,
                                                         ax=ax)
    
    
    ax.tick_params(which='both', length=20,direction="in", bottom=True, top=True,left=True, right=True)
    ax.tick_params(which='minor', length=15)
    
    dats_r.append(dats)
    fig.savefig(f"massive_stars/M83/CMD_r_{rs[i+1]}.png")
    plt.close(fig)

In [ ]:
reds_r = []
blues_r = []
for r, dats in zip(rs[1:], dats_r):
    red = []
    blue = []
    for i, dat in enumerate(dats):
        x = dat[0]
        y = dat[1]
        ind_r = (x>=0.75-col_diff) &  (x<=1.5-col_diff)
        ind_b = (x>=0.-col_diff) &  (x<=.5-col_diff)
        red.extend(list(y[ind_r]))
        blue.extend(list(y[ind_b]))

    bins = np.arange(20,24,0.5)-mag_diff
    yr, xr = np.histogram(red, bins=bins)
    yb, xb = np.histogram(blue, bins=bins)
    xr = 0.5*(xr[1:] + xr[:-1])
    reds_r.append(yr)
    blues_r.append(yb)

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]
    ax.plot(x,y, lw=3, label=f"""r = {rs[1:][i]}" """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Red ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/M83/Red_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = blues_r[i]
    ax.plot(x,y, lw=3)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Blue ${\rm LF_{F115W}}$')
ax.set_yscale('log');
fig.savefig(f"massive_stars/M83/Blue_F115W_LFs.png", bbox_inches='tight');

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]/blues_r[i]
    ax.plot(x,y,lw=1,label=f"""r = {rs[1:][i]}" """)

arr = np.array(reds_sim)/np.array(blues_sim)
for i in range(arr.shape[1]):
    ax.plot([x[i]]*arr.shape[0], arr[:,i],color='black')
    
markers = ['o','D']
for i in range(len(blues_sim)):
    y = reds_sim[i]/blues_sim[i]
    ax.scatter(x,y, s=30, marker=markers[i],lw=3,label=f"""Z = {Zs[i]} """, zorder=300)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel('\#Red/\#Blue')
ax.legend()
ax.set_yscale('log')
fig.savefig(f"massive_stars/M83/Blue_Red.png", bbox_inches='tight');

In [ ]:
for r, dats in zip(rs[1:], dats_r):
    fig, ax = plt.subplots(len(dats),1, figsize=(16,30), sharex=True)
    for i, dat in enumerate(dats):
        if len(dat[0])<10:
            bins  = np.arange(x_cut_settings['x_lo'],x_cut_settings['x_hi'], 0.05)
        else:
            bins  = np.arange(dat[0].min(), dat[0].max(), 0.05)
            
        y, x = np.histogram(dat[0], bins=bins, density=True)
        x = 0.5*(x[1:] + x[:-1])
        ax[i].step(x,y, where='mid', lw=3,color='black')

        if r <= 80:
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.02_mag{y_bins[i]+mag_diff}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='red')
        
        else :
            dat = np.load(f"massive_stars/simulations/col_hist_Z0.008_mag{y_bins[i]+mag_diff}.npy")
            y_, x_ = np.histogram(dat[0], bins=bins, density=True)
            x_ = 0.5*(x_[1:] + x_[:-1])
            ax[i].fill_between(x_,y_, step='mid', lw=3,color='green')

        ymax = np.nanmax([y.max(), y_.max()])
        if np.isnan(ymax):
            ymax=1/1.5
            
        ax[i].annotate(f'{y_bins[i]+mag_diff}-{y_bins[i+1]+mag_diff}', 
                       (1.3,1.1*ymax), color='red')
    
        ax[i].xaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].yaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
        ax[i].tick_params(which='minor', length=8)
        ax[i].minorticks_on()
        ax[i].set_ylabel('Counts')


        ax[i].set_ylim(0,1.5*ymax)
        ax[i].set_xlim(-0.5,1.7)
        #ax[i].set_yscale('log')

    ax[i].set_xlabel(r'${\rm F115W - F200W}$')
    plt.subplots_adjust(hspace=0)
    fig.savefig(f"massive_stars/M83/hist_r_{r}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
hdul = fits.open('../data/M83/jw01783-o002_t010_miri_f770w/jw01783-o002_t010_miri_f770w_i2d.fits')
f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

In [ ]:

fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=f770w_wcs)
norm = simple_norm(f770w, 'log', vmin=-0.1, vmax=15, log_a=10)

ax.imshow(f770w, cmap='gray', norm=norm)

x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
img = ax.scatter(x,y,s=50, color='red', transform=ax.get_transform('icrs'), edgecolor='red', linewidth=0.2, )
ax.set_xlim(350,3450)
ax.set_ylim(0,1050)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$', fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

fig.savefig("massive_stars/M83/RSG_spatial.png", bbox_inches='tight')
plt.close(fig);

# **NGC4449**

In [ ]:
col_diff  = (Av_dict['f115w'] - Av_dict['f200w'])*regions_dict['ngc4449']['Av']
mag_diff  = regions_dict['ngc4449']['dismod'] + (Av_dict['f115w'])*regions_dict['ngc4449']['Av']

ngc_4449_corr = np.array([col_diff, mag_diff])

trgb_polygon_ngc4449 = shapely.Polygon(trgb_box + ngc_4449_corr)
rsg_polygon_ngc4449  = shapely.Polygon(rsg_box + ngc_4449_corr)
rrsg_polygon_ngc4449 = shapely.Polygon(rrsg_box + ngc_4449_corr)
ysg_polygon_ngc4449  = shapely.Polygon(ysg_box + ngc_4449_corr)
bsg_polygon_ngc4449  = shapely.Polygon(bsg_box + ngc_4449_corr)
yms_polygon_ngc4449  = shapely.Polygon(yms_box + ngc_4449_corr)



In [ ]:
col_diff, mag_diff   =  ngc_628_corr - ngc_4449_corr

In [ ]:
mag_diff

In [ ]:
tab = Table.read('../photometry/ngc4449/f115w_f150w_f200w_photometry.fits')

fig, ax = plt.subplots(figsize=(15,12))
ra_cen = regions_dict['ngc4449']['ra']
dec_cen =regions_dict['ngc4449']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

region = {'r_in':0,
          'r_out': 2000,
          'spatial_filter': 'circle'}

extinction = {'Av'  : regions_dict['ngc4449']['Av'],
              'Av_x': 1.8,
              'Av_y': 19,
              'Av_' : 2}

axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
              'ylims': [18-mag_diff, 28-mag_diff]}

isochrone_params={'met': [0.008],
                  'label_min': 0,
                  'label_max': 10,
                  'ages': [6.7, 6.8, 7,7.4,7.7, 8]}

error_settings = {'ref_xpos': -0.4,
                  'mag_err_lim':0.2}

plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}

df_cmd_jwst_n = pd.read_csv('../data/isochrones_master/cmd_jwst_master.csv')
df_temp = df_cmd_jwst_n.copy()
df_temp = df_temp[((df_temp['logAge']==6.7) & (df_temp['label']<5)) | (df_temp['logAge']>6.7)]

fig,ax, tab1 = gen_CMD(tab, 
                      df_temp,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['ngc4449']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': False, 'skip_data': False},
                      fig=fig, ax=ax)

isochrone_params['ages'] = [9.0, 9.4, 10.]
isochrone_params['met'] = [0.002]

fig,ax, tab1 = gen_CMD(tab, 
                      df_cmd_jwst,
                      filters, 
                      positions,
                      region,
                      extinction,
                      regions_dict['ngc4449']['dismod'],
                      axis_limits,
                      isochrone_params,
                      plot_settings=plot_settings,
                      error_settings=error_settings,
                      other_settings={'ab_dist': True, 'skip_data': True},
                      fig=fig, ax=ax)


x = tab1['mag_vega_F115W'] - tab1['mag_vega_F200W']
y = tab1['mag_vega_F115W']

points = np.column_stack([x, y])

ind_trgb = shapely.contains_xy(trgb_polygon_ngc4449, points)
ind_rsg = shapely.contains_xy(rsg_polygon_ngc4449, points)
ind_rrsg = shapely.contains_xy(rrsg_polygon_ngc4449, points)
ind_ysg = shapely.contains_xy(ysg_polygon_ngc4449, points)
ind_bsg = shapely.contains_xy(bsg_polygon_ngc4449, points)
ind_yms = shapely.contains_xy(yms_polygon_ngc4449, points)

tab_TRGB = tab1[ind_trgb]
tab_RSG  = tab1[ind_rsg]
tab_RRSG = tab1[ind_rrsg]
tab_YSG  = tab1[ind_ysg]
tab_BSG  = tab1[ind_bsg]
tab_YMS  = tab1[ind_yms]

ax.scatter(x[ind_rsg], y[ind_rsg], color='red', s=3)
ax.scatter(x[ind_rrsg], y[ind_rrsg], color='magenta', s=3)
ax.scatter(x[ind_ysg], y[ind_ysg], color='yellow', s=3)
ax.scatter(x[ind_bsg], y[ind_bsg], color='cyan', s=3)
ax.scatter(x[ind_yms], y[ind_yms], color='purple', s=3)

#ax.scatter(x[ind_rsg], y[ind_rsg], color='red', s=3)

ax.legend(fontsize=20)
fig.savefig('massive_stars/NGC4449/NGC4449_CMD.png', bbox_inches='tight')

In [ ]:
tab_RSG.write('ngc4449_tables/RSG.fits', overwrite=True)
tab_RRSG.write('ngc4449_tables/RRSG.fits', overwrite=True)
tab_YSG.write('ngc4449_tables/YSG.fits', overwrite=True)
tab_BSG.write('ngc4449_tables/BSG.fits', overwrite=True)
tab_YMS.write('ngc4449_tables/YMS.fits', overwrite=True)

In [ ]:
text = """# Region file format: DS9 version 4.1
global color=green dashlist=8 3 width=1 font="helvetica 10 normal roman" select=1 highlite=1 dash=0 fixed=0 edit=1 move=1 delete=1 include=1 source=1
icrs
"""

# RSG
with open('ngc4449_tables/rsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_RSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=red \n""")

with open('ngc4449_tables/rsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_RSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=red point=circle \n""")

# RRSG
with open('ngc4449_tables/rrsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_RRSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=magenta \n""")

with open('ngc4449_tables/rrsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_RRSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=magenta point=circle \n""")

# YSG
with open('ngc4449_tables/ysg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_YSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=yellow \n""")

with open('ngc4449_tables/ysg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_YSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=yellow point=circle \n""")

# BSG
with open('ngc4449_tables/bsg_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_BSG:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=cyan \n""")

with open('ngc4449_tables/bsg_point.reg','w') as f:
    f.writelines(text)
    for row in tab_BSG:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=cyan point=circle \n""")

# YMS
with open('ngc4449_tables/yms_circle.reg','w') as f:
    f.writelines(text)
    for row in tab_YMS:
        f.writelines(f"""circle({np.round(row['ra'],7)}, {np.round(row['dec'],7)},0.1") #color=purple \n""")

with open('ngc4449_tables/yms_point.reg','w') as f:
    f.writelines(text)
    for row in tab_YMS:
        f.writelines(f"""point({np.round(row['ra'],7)}, {np.round(row['dec'],7)}) #color=purple point=circle \n""")

        

In [ ]:
tab_RSG = tab1[ind_rsg]
r = angular_separation(tab_RSG['ra']*u.deg, tab_RSG['dec']*u.deg ,regions_dict['ngc4449']['ra']*u.deg, regions_dict['ngc4449']['dec']*u.deg).to(u.arcsec).value

In [ ]:
r.max()

In [ ]:
x= tab_RSG['ra']
y = tab_RSG['dec']
c= r

img = plt.scatter(x,y,c=c, cmap='tab10',s=1, vmin=0, vmax = 191)
cb = plt.colorbar(img)
cb.set_ticks([0,20,40,50,60,80,100,150,200])

In [ ]:
rs = [0,40,198]

In [ ]:
tab     = Table.read('../photometry/ngc4449/f115w_f150w_f200w_photometry.fits')

dats_r  = []
ra_cen  = regions_dict['ngc4449']['ra']
dec_cen = regions_dict['ngc4449']['dec']

filters = {'filt1':'f115w',
           'filt2':'f200w',
           'filt3':'f115w'}

positions = {'ra_col' : 'ra',
             'dec_col': 'dec',
             'ra_cen' : ra_cen,
             'dec_cen': dec_cen}

for i in range(len(rs)-1):

    region = {'a1': rs[i],
              'b1': rs[i],
              'a2': rs[i+1],
              'b2': rs[i+1],
              'ang' : 0,
              'spatial_filter': 'ellipse'}
    
    extinction = {'Av'  : regions_dict['ngc4449']['Av'],
                  'Av_x': 1.8,
                  'Av_y': 19,
                  'Av_' : 2}
    
    axis_limits= {'xlims': [-0.5-col_diff, 3-col_diff], 
                  'ylims': [18-mag_diff, 28-mag_diff]}
    
    isochrone_params={'met': 0.002,
                      'label_min': 0,
                      'label_max': 3,
                      'age': 10.0}
    
    error_settings = {'ref_xpos': -0.4,
                      'mag_err_lim':0.2}
    
    plot_settings = {'s':2, 'legend.ncols':2, 'alpha':0.7, 'lw':3}
    
    m = 21.35#f200w[k]
    
    x_cut_settings = {'cmd_xlo' : -1,
                  'cmd_xhi' : 1.6,
                  'cmd_ylo' : 22,
                  'cmd_yhi' : 26,
    
                  'rgb_xlo' : -1,
                  'rgb_xhi' : 1.6,
                  'rgb_ylo' : 21,
                  'rgb_yhi' : 24,
                  'y_lo'    : 20  - mag_diff,
                  'y_hi'    : 24.1- mag_diff ,
                  'x_lo'    : -1,
                  'x_hi'    : 1.6,                  
                  'dy'      : 0.5,
                  'dx'      : 2.,
                  'theta'   : np.radians(-50),
                  'slope'   : 0,
                  'intercept' : 0.6-col_diff,
                  'fit_rgb' : False,
                  'fit_isochrone' : False,
                  'rgb_yhi' : 25.5,
                  'color'   : 'black'}
        
    fig, ax = plt.subplots(figsize=(12, 12))
    
    _, _, dats, x_val, y_val, y_bins, model_rgb= gen_CMD_xcut(tab=tab,
                                                         df_iso=df_cmd_jwst,
                                                         distance_modulus=regions_dict['ngc4449']['dismod'],
                                                         filters=filters,
                                                         positions=positions,
                                                         region=region,
                                                         extinction=extinction,
                                                         axis_limits=axis_limits,
                                                         isochrone_params=isochrone_params,
                                                         error_settings=error_settings,
                                                         plot_settings=plot_settings,
                                                         x_cut_settings=x_cut_settings,
                                                         fig=fig,
                                                         ax=ax)
    
    
    ax.tick_params(which='both', length=20,direction="in", bottom=True, top=True,left=True, right=True)
    ax.tick_params(which='minor', length=15)
    
    dats_r.append(dats)
    fig.savefig(f"massive_stars/NGC4449/CMD_r_{rs[i+1]}.png")
    plt.close(fig)

In [ ]:
reds_r = []
blues_r = []
for r, dats in zip(rs[1:], dats_r):
    red = []
    blue = []
    for i, dat in enumerate(dats):
        x = dat[0]
        y = dat[1]
        ind_r = (x>=0.75-col_diff) &  (x<=1.5-col_diff)
        ind_b = (x>=0.-col_diff) &  (x<=.5-col_diff)
        red.extend(list(y[ind_r]))
        blue.extend(list(y[ind_b]))

    bins = np.arange(20,24,0.5)-mag_diff
    yr, xr = np.histogram(red, bins=bins)
    yb, xb = np.histogram(blue, bins=bins)
    xr = 0.5*(xr[1:] + xr[:-1])
    reds_r.append(yr)
    blues_r.append(yb)

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]
    ax.plot(x,y, lw=3, label=f"""r = {rs[1:][i]}" """)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Red ${\rm LF_{F115W}}$')
ax.set_yscale('log')
ax.legend()
fig.savefig(f"massive_stars/NGC4449/Red_F115W_LFs.png", bbox_inches='tight');


In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = blues_r[i]
    ax.plot(x,y, lw=3)

ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel(r'Blue ${\rm LF_{F115W}}$')
ax.set_yscale('log');
fig.savefig(f"massive_stars/NGC4449/Blue_F115W_LFs.png", bbox_inches='tight');

In [ ]:
fig, ax = plt.subplots(figsize=(15,5))
bins = np.arange(20,24,0.5)
x =  0.5*(bins[1:] + bins[:-1])

for i in range(len(reds_r)):
    y = reds_r[i]/blues_r[i]
    ax.plot(x,y,lw=1,label=f"""r = {rs[1:][i]}" """)

arr = np.array(reds_sim)/np.array(blues_sim)
for i in range(arr.shape[1]):
    ax.plot([x[i]]*arr.shape[0], arr[:,i],color='black')
    
markers = ['o','D']
for i in range(len(blues_sim)):
    y = reds_sim[i]/blues_sim[i]
    ax.scatter(x,y, s=30, marker=markers[i],lw=3,label=f"""Z = {Zs[i]} """, zorder=300)



ax.xaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_major_locator(AutoLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())
ax.tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
ax.tick_params(which='minor', length=8)
ax.minorticks_on()

ax.set_xlabel(r'${\rm F115W}$')
ax.set_ylabel('\#Red/\#Blue')
ax.legend()
ax.set_yscale('log')
fig.savefig(f"massive_stars/NGC4449/Blue_Red.png", bbox_inches='tight');

In [ ]:
for r, dats in zip(rs[1:], dats_r):
    fig, ax = plt.subplots(len(dats),1, figsize=(16,30), sharex=True)
    for i, dat in enumerate(dats):
        if len(dat[0])<10:
            bins  = np.arange(x_cut_settings['x_lo'],x_cut_settings['x_hi'], 0.05)
        else:
            bins  = np.arange(dat[0].min(), dat[0].max(), 0.05)
            
        y, x = np.histogram(dat[0], bins=bins, density=True)
        x = 0.5*(x[1:] + x[:-1])
        ax[i].step(x,y, where='mid', lw=3,color='black')


        dat = np.load(f"massive_stars/simulations/col_hist_Z0.008_mag{y_bins[i]+mag_diff}.npy")
        y_, x_ = np.histogram(dat[0], bins=bins, density=True)
        x_ = 0.5*(x_[1:] + x_[:-1])
        ax[i].fill_between(x_,y_, step='mid', lw=3,color='green')

        ymax = np.nanmax([y.max(), y_.max()])
        if np.isnan(ymax):
            ymax=1/1.5
            
        ax[i].annotate(f'{y_bins[i]+mag_diff}-{y_bins[i+1]+mag_diff}', 
                       (1.3,1.1*ymax), color='red')
    
        ax[i].xaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].yaxis.set_major_locator(AutoLocator())
        ax[i].yaxis.set_minor_locator(AutoMinorLocator())
        ax[i].tick_params(which='both', length=15,direction="in", bottom=True, top=True,left=True, right=True)
        ax[i].tick_params(which='minor', length=8)
        ax[i].minorticks_on()
        ax[i].set_ylabel('Counts')


        ax[i].set_ylim(0,1.5*ymax)
        ax[i].set_xlim(-0.5,1.7)
        #ax[i].set_yscale('log')

    ax[i].set_xlabel(r'${\rm F115W - F200W}$')
    plt.subplots_adjust(hspace=0)
    fig.savefig(f"massive_stars/NGC4449/hist_r_{r}.png", bbox_inches='tight')
    plt.close(fig)

In [ ]:
hdul = fits.open('../data/NGC4449/jw01783-c1016_t012_miri_f770w_i2d.fits')
f770w = hdul[1].data
f770w_wcs = WCS(hdul[1].header)

In [ ]:
f770w.shape

In [ ]:
fig = plt.figure(figsize=(30,20))
ax = fig.add_subplot(projection=f770w_wcs)
norm = simple_norm(f770w, 'log', vmin=-0.1, vmax=15, log_a=10)

ax.imshow(f770w, cmap='gray', norm=norm)

x,y = tab_RSG['ra'].value.astype(np.float64), tab_RSG['dec'].value.astype(np.float64)
img = ax.scatter(x,y,s=50, color='red', transform=ax.get_transform('icrs'), edgecolor='red', linewidth=0.2, )
ax.set_xlim(350,2317)
ax.set_ylim(0,1050)

ax.set_xlabel(r'${\rm Dec \,(J2000)}$', fontsize=30)
ax.set_ylabel(r'${\rm RA \,(J2000)}$', fontsize=30)

ax.xaxis.set_minor_locator(AutoMinorLocator())
ax.yaxis.set_minor_locator(AutoMinorLocator())

fig.savefig("massive_stars/NGC4449/RSG_spatial.png", bbox_inches='tight')
plt.close(fig);